## STEP 1 — UPDATE SYSTEM

In [ ]:
sudo apt update
sudo apt full-upgrade -y
sudo reboot
uname -r  # verify kernel

## STEP 2 — INSTALL PACKAGES

In [ ]:
sudo apt install -y qemu-kvm libvirt-daemon-system libvirt-clients virtinst libguestfs-tools cloud-image-utils genisoimage openvswitch-switch openvswitch-common lvm2 thin-provisioning-tools gdisk parted bridge-utils dnsmasq iptables iptables-persistent netfilter-persistent curl wget git vim htop iotop net-tools tcpdump

## STEP 3 — CONFIG SERVICES

In [ ]:
sudo usermod -aG libvirt,kvm $USER  # relogin after
sudo systemctl enable --now libvirtd

virsh net-destroy default || true
virsh net-undefine default || true
virsh net-list --all  # should be empty

sudo systemctl enable --now openvswitch-switch
sudo ovs-vsctl show  # verify OVS

## STEP 4 — NESTED KVM

In [ ]:
cat /sys/module/kvm_intel/parameters/nested  # N/Y
echo "options kvm-intel nested=1" | sudo tee /etc/modprobe.d/kvm-nested.conf
sudo modprobe -r kvm_intel || true
sudo modprobe kvm_intel
cat /sys/module/kvm_intel/parameters/nested  # expect Y

## STEP 5 — IOMMU

In [ ]:
sudo nano /etc/default/grub
# set: GRUB_CMDLINE_LINUX="intel_iommu=on iommu=pt"
sudo update-grub
sudo reboot
dmesg | grep Intel-IOMMU
ls /sys/kernel/iommu_groups/ | wc -l  # >0 OK

## STEP 6 — LVM THIN POOL (careful chọn disk)

In [ ]:
lsblk -d -o NAME,SIZE,TYPE,MOUNTPOINT  # do NOT use disk with '/'
# Example: use /dev/sdb for LVM
sudo pvcreate /dev/sdb
sudo vgcreate vg-lab /dev/sdb
sudo lvcreate --type thin-pool --extents 95%VG --name thin-pool vg-lab
sudo lvs  # expect thin pool

# libvirt pool
cat > /tmp/pool-vg-lab.xml << 'EOF'
<pool type="logical">
  <name>vm-pool</name>
  <source>
    <name>vg-lab</name>
    <format type="lvm2"/>
  </source>
  <target>
    <path>/dev/vg-lab</path>
  </target>
</pool>
EOF

virsh pool-define /tmp/pool-vg-lab.xml
virsh pool-start vm-pool
virsh pool-autostart vm-pool
virsh pool-list --all  # expect active

# (optional) Ceph OSD on separate disk, e.g. /dev/sdc
# sudo wipefs -a /dev/sdc
# sudo ceph orch daemon add osd server-a:/dev/sdc